In [ ]:
import pandas as pd
import numpy as np
from models import cmre, nmre, ndee, ocsvm
from plots import plot_appendix, plot_main_paper
import copy
import pickle

In [ ]:
RESULTS_DIR = ''

# Simulation 5 Experiments (Appendix F.2.5)

In [ ]:
def generate_sgc_datasets(mr,
                          sa_normal,
                          sa_untrustworthy,
                          sa_mediator,
                          te
                          ):

    # Load dataset
    df = pd.read_excel('./datasets/default_of_credit_card_clients.xls', skiprows=1)
    df = df.sample(frac=1).reset_index(drop=True)
    

    # Drop rows that are not needed
    df = df[['SEX', 'EDUCATION', 'MARRIAGE', 'AGE']]

    # Convert sex to binary variable
    df['SEX'] = df['SEX'] - 1

    # Convert marriage to binary variable
    df['MARRIAGE'] = np.where(df['MARRIAGE'] > 1, 1, 0)

    # Convert education to binary variable
    df['EDUCATION'] = np.where(df['EDUCATION'] > 2, 1, 0)

    # Min-max scale the data
    df['AGE'] = (df['AGE'] - df['AGE'].min()) / (df['AGE'].max() - df['AGE'].min())
    
    # Use marriage as selection bias
    agent_prob = 0.05  + (1-df['MARRIAGE'])*0.4
    df['AGENT'] = np.random.binomial(1, agent_prob, len(df))
    
    # Have agent strategically manipulate education
    df['EDUCATION'] = df['EDUCATION'] + (1-df['EDUCATION'])*df['AGENT']*np.random.binomial(1, sa_mediator, len(df))
    

    # Generate employment variable
    employment_prob = 0.05 \
                    + df['MARRIAGE']*0.2 \
                    + df['EDUCATION']*df['SEX']*0.2 \
                    + np.square(df['AGE'])*0.05 \
                    + df['AGENT'] * sa_untrustworthy + (1-df['AGENT'])*sa_normal
    df['EMPLOYMENT'] = np.random.binomial(1, employment_prob, len(df))

    # Generate default variable
    default_prob = 0.05 \
                    + df['SEX']*0.3 \
                    + np.square(df['AGE'])*0.05 \
                    + df['EMPLOYMENT'] * te
    df['DEFAULT'] = np.random.binomial(1, default_prob, len(df))

    # Strategically misreport the dataset (misreported employment status)
    prob_agent_employment = len(df[(df['AGENT'] == 1) & (df['EMPLOYMENT'] == 1)]) / len(df[df['AGENT'] == 1])
    df['EMPLOYMENT'] = df['EMPLOYMENT'] + df['AGENT']*(1-df['EMPLOYMENT']) * np.random.binomial(1, ((employment_prob / (1-mr)) - employment_prob) / (1 - employment_prob),  len(df))

    return df

# Results for semi-synthetic loan experiments

### Estimate MR given different causal effects

In [ ]:
num_sims = 100
causal_effects = [0.1, 0.2, 0.3, 0.4, 0.5]
# Dataframes to keep track of results
cmre_loan_results_ce_df_normal = pd.DataFrame(columns=['sim_num', 'ce', 'mr'])
nmre_loan_results_ce_df_normal = pd.DataFrame(columns=['sim_num', 'ce', 'mr'])
ndee_all_loan_results_ce_df_normal = pd.DataFrame(columns=['sim_num', 'ce', 'mr'])
ndee_no_s_loan_results_ce_df_normal = pd.DataFrame(columns=['sim_num', 'ce', 'mr'])
ndee_no_c_loan_results_ce_df_normal = pd.DataFrame(columns=['sim_num', 'ce', 'mr'])
ocsvm_loan_results_ce_df_normal = pd.DataFrame(columns=['sim_num', 'ce', 'mr'])

# Perform a few simulations for each method
for sim in range(num_sims):
    # Set the random seed
    np.random.seed(sim)
    
    for causal_effect in causal_effects:

        # Generate dataset for simulation
        df = generate_sgc_datasets(0.2, 0.0, 0.1, 0.2, causal_effect)
        normal_dataset = df[df['AGENT'] == 0]
        strategic_dataset = df[df['AGENT'] == 1]
        
        # Get misreporting rates and keep track of results
        mr = cmre('EMPLOYMENT', 'DEFAULT', ['SEX', 'AGE'], normal_dataset, strategic_dataset)
        cmre_loan_results_ce_df_normal.loc[len(cmre_loan_results_ce_df_normal)] = [sim, causal_effect, mr]

        mr = nmre('EMPLOYMENT', 'DEFAULT', normal_dataset, strategic_dataset)
        nmre_loan_results_ce_df_normal.loc[len(nmre_loan_results_ce_df_normal)] = [sim, causal_effect, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['EDUCATION', 'SEX', 'MARRIAGE', 'AGE'], df)
        ndee_all_loan_results_ce_df_normal.loc[len(ndee_all_loan_results_ce_df_normal)] = [sim, causal_effect, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['EDUCATION', 'SEX', 'AGE'], df)
        ndee_no_s_loan_results_ce_df_normal.loc[len(ndee_no_s_loan_results_ce_df_normal)] = [sim, causal_effect, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['MARRIAGE', 'EDUCATION'], df)
        ndee_no_c_loan_results_ce_df_normal.loc[len(ndee_no_c_loan_results_ce_df_normal)] = [sim, causal_effect, mr]

        mr = ocsvm(normal_dataset, strategic_dataset)
        ocsvm_loan_results_ce_df_normal.loc[len(ocsvm_loan_results_ce_df_normal)] = [sim, causal_effect, mr]

# Get mean and std for each causal effect
cmre_loan_results_ce_df_normal = cmre_loan_results_ce_df_normal.groupby('ce')['mr'].agg(['mean', 'std']).reset_index()
nmre_loan_results_ce_df_normal = nmre_loan_results_ce_df_normal.groupby('ce')['mr'].agg(['mean', 'std']).reset_index()
ndee_all_loan_results_ce_df_normal = ndee_all_loan_results_ce_df_normal.groupby('ce')['mr'].agg(['mean', 'std']).reset_index()
ndee_no_s_loan_results_ce_df_normal = ndee_no_s_loan_results_ce_df_normal.groupby('ce')['mr'].agg(['mean', 'std']).reset_index()
ndee_no_c_loan_results_ce_df_normal = ndee_no_c_loan_results_ce_df_normal.groupby('ce')['mr'].agg(['mean', 'std']).reset_index()
ocsvm_loan_results_ce_df_normal = ocsvm_loan_results_ce_df_normal.groupby('ce')['mr'].agg(['mean', 'std']).reset_index()

In [ ]:
# Save info
loan_list_ce_normal = [cmre_loan_results_ce_df_normal,
                       nmre_loan_results_ce_df_normal,
                       ndee_all_loan_results_ce_df_normal,
                       ndee_no_s_loan_results_ce_df_normal,
                       ndee_no_c_loan_results_ce_df_normal,
                       ocsvm_loan_results_ce_df_normal]
with open(f'{RESULTS_DIR}/loan_list_ce_normal.pkl', 'wb') as f:
    pickle.dump(loan_list_ce_normal, f)

### Estimate MR given different Misrepoting Rates

In [ ]:
num_sims = 100
mr_probs = [0.0, 0.05, 0.10, 0.15, 0.20]

# Dataframes to keep track of results
cmre_loan_results_mr_df_normal = pd.DataFrame(columns=['sim_num', 'true_mr', 'mr'])
nmre_loan_results_mr_df_normal = pd.DataFrame(columns=['sim_num', 'true_mr', 'mr'])
ndee_all_loan_results_mr_df_normal = pd.DataFrame(columns=['sim_num', 'true_mr', 'mr'])
ndee_no_s_loan_results_mr_df_normal = pd.DataFrame(columns=['sim_num', 'true_mr', 'mr'])
ndee_no_c_loan_results_mr_df_normal = pd.DataFrame(columns=['sim_num', 'true_mr', 'mr'])
ocsvm_loan_results_mr_df_normal = pd.DataFrame(columns=['sim_num', 'true_mr', 'mr'])

# Perform a few simulations for each method
for sim in range(num_sims):
    # Set the random seed
    np.random.seed(sim)
    
    for true_mr in mr_probs:

        # Generate dataset for simulation
        df = generate_sgc_datasets(true_mr, 0.0, 0.1, 0.2, 0.4)

        normal_dataset = df[df['AGENT'] == 0]
        strategic_dataset = df[df['AGENT'] == 1]
        
        # Get misreporting rates and keep track of results
        mr = cmre('EMPLOYMENT', 'DEFAULT', ['SEX', 'AGE'], normal_dataset, strategic_dataset)
        cmre_loan_results_mr_df_normal.loc[len(cmre_loan_results_mr_df_normal)] = [sim, true_mr, mr]

        mr = nmre('EMPLOYMENT', 'DEFAULT', normal_dataset, strategic_dataset)
        nmre_loan_results_mr_df_normal.loc[len(nmre_loan_results_mr_df_normal)] = [sim, true_mr, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['EDUCATION', 'SEX', 'MARRIAGE', 'AGE'], df)
        ndee_all_loan_results_mr_df_normal.loc[len(ndee_all_loan_results_mr_df_normal)] = [sim, true_mr, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['EDUCATION', 'SEX', 'AGE'], df)
        ndee_no_s_loan_results_mr_df_normal.loc[len(ndee_no_s_loan_results_mr_df_normal)] = [sim, true_mr, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['MARRIAGE', 'EDUCATION'], df)
        ndee_no_c_loan_results_mr_df_normal.loc[len(ndee_no_c_loan_results_mr_df_normal)] = [sim, true_mr, mr]

        mr = ocsvm(normal_dataset, strategic_dataset)
        ocsvm_loan_results_mr_df_normal.loc[len(ocsvm_loan_results_mr_df_normal)] = [sim, true_mr, mr]
        

# Get mean and std for each causal effect
cmre_loan_results_mr_df_normal = cmre_loan_results_mr_df_normal.groupby('true_mr')['mr'].agg(['mean', 'std']).reset_index()
nmre_loan_results_mr_df_normal = nmre_loan_results_mr_df_normal.groupby('true_mr')['mr'].agg(['mean', 'std']).reset_index()
ndee_all_loan_results_mr_df_normal = ndee_all_loan_results_mr_df_normal.groupby('true_mr')['mr'].agg(['mean', 'std']).reset_index()
ndee_no_s_loan_results_mr_df_normal = ndee_no_s_loan_results_mr_df_normal.groupby('true_mr')['mr'].agg(['mean', 'std']).reset_index()
ndee_no_c_loan_results_mr_df_normal = ndee_no_c_loan_results_mr_df_normal.groupby('true_mr')['mr'].agg(['mean', 'std']).reset_index()
ocsvm_loan_results_mr_df_normal = ocsvm_loan_results_mr_df_normal.groupby('true_mr')['mr'].agg(['mean', 'std']).reset_index()

In [ ]:
# Save info
loan_list_mr_normal = [cmre_loan_results_mr_df_normal,
                       nmre_loan_results_mr_df_normal,
                       ndee_all_loan_results_mr_df_normal,
                       ndee_no_s_loan_results_mr_df_normal,
                       ndee_no_c_loan_results_mr_df_normal,
                       ocsvm_loan_results_mr_df_normal]
with open(f'{RESULTS_DIR}/loan_list_mr_normal.pkl', 'wb') as f:
    pickle.dump(loan_list_mr_normal, f)

### Estimate MR given different genuine adaptation rates

In [ ]:
num_sims = 100
sa_probs = [0.0, 0.05, 0.10, 0.15, 0.2, 0.25, 0.3]

# Dataframes to keep track of results
cmre_loan_results_sa_df_normal = pd.DataFrame(columns=['sim_num', 'sa', 'mr'])
nmre_loan_results_sa_df_normal = pd.DataFrame(columns=['sim_num', 'sa', 'mr'])
ndee_all_loan_results_sa_df_normal = pd.DataFrame(columns=['sim_num', 'sa', 'mr'])
ndee_no_s_loan_results_sa_df_normal = pd.DataFrame(columns=['sim_num', 'sa', 'mr'])
ndee_no_c_loan_results_sa_df_normal = pd.DataFrame(columns=['sim_num', 'sa', 'mr'])
ocsvm_loan_results_sa_df_normal = pd.DataFrame(columns=['sim_num', 'sa', 'mr'])

# Perform a few simulations for each method
for sim in range(num_sims):
    # Set the random seed
    np.random.seed(sim)
    
    for sa in sa_probs:

        # Generate dataset for simulation
        df = generate_sgc_datasets(0.2, 0.0, sa, 0.2, 0.4)

        normal_dataset = df[df['AGENT'] == 0]
        strategic_dataset = df[df['AGENT'] == 1]
        
        # Get misreporting rates and keep track of results
        mr = cmre('EMPLOYMENT', 'DEFAULT', ['SEX', 'AGE'], normal_dataset, strategic_dataset)
        cmre_loan_results_sa_df_normal.loc[len(cmre_loan_results_sa_df_normal)] = [sim, sa, mr]

        mr = nmre('EMPLOYMENT', 'DEFAULT', normal_dataset, strategic_dataset)
        nmre_loan_results_sa_df_normal.loc[len(nmre_loan_results_sa_df_normal)] = [sim, sa, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['EDUCATION', 'SEX', 'MARRIAGE', 'AGE'], df)
        ndee_all_loan_results_sa_df_normal.loc[len(ndee_all_loan_results_sa_df_normal)] = [sim, sa, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['EDUCATION', 'SEX', 'AGE'], df)
        ndee_no_s_loan_results_sa_df_normal.loc[len(ndee_no_s_loan_results_sa_df_normal)] = [sim, sa, mr]

        mr = ndee('AGENT', 'EMPLOYMENT', ['MARRIAGE', 'EDUCATION'], df)
        ndee_no_c_loan_results_sa_df_normal.loc[len(ndee_no_c_loan_results_sa_df_normal)] = [sim, sa, mr]

        mr = ocsvm(normal_dataset, strategic_dataset)
        ocsvm_loan_results_sa_df_normal.loc[len(ocsvm_loan_results_sa_df_normal)] = [sim, sa, mr]

# Get mean and std for each causal effect
cmre_loan_results_sa_df_normal = cmre_loan_results_sa_df_normal.groupby('sa')['mr'].agg(['mean', 'std']).reset_index()
nmre_loan_results_sa_df_normal = nmre_loan_results_sa_df_normal.groupby('sa')['mr'].agg(['mean', 'std']).reset_index()
ndee_all_loan_results_sa_df_normal = ndee_all_loan_results_sa_df_normal.groupby('sa')['mr'].agg(['mean', 'std']).reset_index()
ndee_no_s_loan_results_sa_df_normal = ndee_no_s_loan_results_sa_df_normal.groupby('sa')['mr'].agg(['mean', 'std']).reset_index()
ndee_no_c_loan_results_sa_df_normal = ndee_no_c_loan_results_sa_df_normal.groupby('sa')['mr'].agg(['mean', 'std']).reset_index()
ocsvm_loan_results_sa_df_normal = ocsvm_loan_results_sa_df_normal.groupby('sa')['mr'].agg(['mean', 'std']).reset_index()

In [ ]:
# Save info
loan_list_sa_normal = [cmre_loan_results_sa_df_normal,
                       nmre_loan_results_sa_df_normal,
                       ndee_all_loan_results_sa_df_normal,
                       ndee_no_s_loan_results_sa_df_normal,
                       ndee_no_c_loan_results_sa_df_normal,
                       ocsvm_loan_results_sa_df_normal]
with open(f'{RESULTS_DIR}/loan_list_sa_normal.pkl', 'wb') as f:
    pickle.dump(loan_list_sa_normal, f)

# Create plots

In [ ]:
# Load lists
with open(f'{RESULTS_DIR}/loan_list_ce_normal.pkl', 'rb') as f:
    loan_list_ce_normal = pickle.load(f)
with open(f'{RESULTS_DIR}/loan_list_mr_normal.pkl', 'rb') as f:
    loan_list_mr_normal = pickle.load(f)
with open(f'{RESULTS_DIR}/loan_list_sa_normal.pkl', 'rb') as f:
    loan_list_sa_normal = pickle.load(f)

In [ ]:
# Create and show plot
synth_list = [loan_list_sa_normal, loan_list_ce_normal, loan_list_mr_normal]
synth_mr_list = [0.2, 0.2, 0.2]
label_list = ['CMRE (Ours)', 'NMRE', 'NDEE (All)', 'NDEE (no S)', 'NDEE (no C)', 'OCSVM']
x_var_list = ['sa', 'ce', 'true_mr']
x_label_list = [r'\textbf{$\beta_{A}$}', r'\textbf{Causal Effect of $X^{*}$ on $Y$}', r'\textbf{Misreporting Rate}']
y_label = r'\textbf{Estimated MR}'

plt = plot_appendix(synth_list,
                label_list,
                synth_mr_list,
                x_var_list,
                x_label_list,
                y_label)


# plt.show()
plt.savefig(f'{RESULTS_DIR}/simulation_5_appendix_plot.pdf', dpi=600, bbox_inches='tight')

In [ ]:
# Create and show plot
indices = [0,1,2,5]
loan_list_sa_normal_small = [loan_list_sa_normal[i] for i in indices]
loan_list_ce_normal_small = [loan_list_ce_normal[i] for i in indices]
loan_list_mr_normal_small = [loan_list_mr_normal[i] for i in indices]
synth_list = [loan_list_sa_normal_small, loan_list_ce_normal_small, loan_list_mr_normal_small]
synth_mr_list = [0.2, 0.2, 0.2]
label_list = [r'\textbf{CMRE (Ours)}', r'\textbf{NMRE}', r'\textbf{NDEE}', r'\textbf{OC-SVM}']
x_var_list = ['sa', 'ce', 'true_mr']
x_label_list = [r'\textbf{$\beta_{A}$}', r'\textbf{Causal Effect of $X^{*}$ on $Y$}', r'\textbf{Misreporting Rate}']
y_label = r'\textbf{Estimated MR}'

plt = plot_main_paper(synth_list,
                label_list,
                synth_mr_list,
                x_var_list,
                x_label_list,
                y_label)

plt.savefig(f'{RESULTS_DIR}/simulation_5_main_plot.pdf', dpi=600, bbox_inches='tight')